In [31]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import random

In [32]:
season_features_m = pd.read_csv('season_features_m.csv')
season_features_w = pd.read_csv('season_features_w.csv')

In [33]:

seeds_m = pd.read_csv('useful_data/MNCAATourneySeeds.csv')
seeds_w = pd.read_csv('useful_data/WNCAATourneySeeds.csv')

tourney_m = pd.read_csv('useful_data/MNCAATourneyDetailedResults.csv')
tourney_w = pd.read_csv('useful_data/WNCAATourneyDetailedResults.csv')




In [34]:


random.seed(52)


temp_m = tourney_m[["Season", "WTeamID","LTeamID"]].copy()
for i in range(len(temp_m)):
    pair = random.choice([("WTeamID","LTeamID"), ("LTeamID","WTeamID")])
    pair = list(pair)
    temp_m.loc[i,["WTeamID","LTeamID"]] = temp_m.loc[i,pair].values

temp_w = tourney_w[["Season", "WTeamID","LTeamID"]].copy()
for i in range(len(temp_w)):
    pair = random.choice([("WTeamID","LTeamID"), ("LTeamID","WTeamID")])
    pair = list(pair)
    temp_w.loc[i,["WTeamID","LTeamID"]] = temp_w.loc[i,pair].values



In [35]:


merged_m = season_features_m.merge(
    seeds_m,
    how='left',          # left join
    on=['Season', 'TeamID']  # kolumny, po których łączymy
)

merged_w = season_features_w.merge(
    seeds_w,
    how='left',          # left join
    on=['Season', 'TeamID']  # kolumny, po których łączymy
)

In [36]:
# merge dla zwycięzców (WTeamID)
m_tourney_with_win = temp_m.merge(
    merged_m,
    how='left',
    left_on=['Season', 'WTeamID'],
    right_on=['Season', 'TeamID'],
    suffixes=('', '_1')  # dodajemy "_w" do kolumn z merged, aby rozróżnić
)

# merge dla przegranych (LTeamID)
m_tourney_full = m_tourney_with_win.merge(
    merged_m,
    how='left',
    left_on=['Season', 'LTeamID'],
    right_on=['Season', 'TeamID'],
    suffixes=('', '_2')  # dodajemy "_l" do kolumn z merged, aby rozróżnić
)

In [37]:
# merge dla zwycięzców (WTeamID)
w_tourney_with_win = temp_w.merge(
    merged_w,
    how='left',
    left_on=['Season', 'WTeamID'],
    right_on=['Season', 'TeamID'],
    suffixes=('', '_1')  # dodajemy "_w" do kolumn z merged, aby rozróżnić
)

# merge dla przegranych (LTeamID)
w_tourney_full = w_tourney_with_win.merge(
    merged_w,
    how='left',
    left_on=['Season', 'LTeamID'],
    right_on=['Season', 'TeamID'],
    suffixes=('', '_2')  # dodajemy "_l" do kolumn z merged, aby rozróżnić
)

In [38]:

m_tourney_full['Is1Winner'] = (m_tourney_full['WTeamID'] == tourney_m['WTeamID']).astype(int)
w_tourney_full['Is1Winner'] = (w_tourney_full['WTeamID'] == tourney_w['WTeamID']).astype(int)
m_tourney_full['1TeamID'] = m_tourney_full['WTeamID']
m_tourney_full['2TeamID'] = m_tourney_full['LTeamID']
w_tourney_full['1TeamID'] = w_tourney_full['WTeamID']
w_tourney_full['2TeamID'] = w_tourney_full['LTeamID']

m_tourney_full.drop(['WTeamID', 'LTeamID', 'TeamID'], axis=1, inplace=True)
w_tourney_full.drop(['WTeamID', 'LTeamID', 'TeamID'], axis=1,inplace=True)

In [39]:
for col in ['Seed', 'Seed_2']:
    m_tourney_full[col] = (
        m_tourney_full[col]
        .str.extract(r'(\d+)')      # wyciąga cyfry
        .astype(float)              # żeby NaN działał
        .fillna(17)                 # brak seed -> 17
        .astype(int)
    )
    w_tourney_full[col] = (
        w_tourney_full[col]
        .str.extract(r'(\d+)')      # wyciąga cyfry
        .astype(float)              # żeby NaN działał
        .fillna(17)                 # brak seed -> 17
        .astype(int)
    )

    # normalizacja + odwrócenie
    m_tourney_full[col] = (17 - m_tourney_full[col]) / 16
    w_tourney_full[col] = (17 - w_tourney_full[col]) / 16

In [40]:
# zapis do pliku CSV
m_tourney_full.to_csv("m_tourney_full.csv", index=False)
w_tourney_full.to_csv("w_tourney_full.csv", index=False)

In [41]:
# Train: sezony do 2023 włącznie
train_m = m_tourney_full[m_tourney_full['Season'] <= 2023]

# Test: sezon 2024
test_m = m_tourney_full[m_tourney_full['Season'] == 2024]

# Validation: sezon 2025
val_m = m_tourney_full[m_tourney_full['Season'] == 2025]

# opcjonalnie podgląd liczby wier

In [42]:

train_m.to_csv("tourney_train_m.csv", index=False)

test_m.to_csv("tourney_test_m.csv", index=False)

val_m.to_csv("tourney_val_m.csv", index=False)


In [43]:
# Train: sezony do 2023 włącznie
train_w = w_tourney_full[w_tourney_full['Season'] <= 2023]

# Test: sezon 2024
test_w = w_tourney_full[w_tourney_full['Season'] == 2024]

# Validation: sezon 2025
val_w = w_tourney_full[w_tourney_full['Season'] == 2025]

# opcjonalnie podgląd liczby wier

train_w.to_csv("tourney_train_w.csv", index=False)

test_w.to_csv("tourney_test_w.csv", index=False)

val_w.to_csv("tourney_val_w.csv", index=False)


# Sekcja z danymi do predykcji

In [44]:
submission = pd.read_csv('SampleSubmission.csv')


submission[['Season', '1TeamID', '2TeamID']] = (
    submission['ID']
    .str.split('_', expand=True)
)


submission[['Season', '1TeamID', '2TeamID']] = (
    submission[['Season', '1TeamID', '2TeamID']].astype(int)
)



In [45]:
# gender split

submission_m = submission[submission['1TeamID'] < 3000]
submission_w = submission[submission['1TeamID'] >= 3000]

In [46]:
# merge dla 1TeamID
submission_m = submission_m.merge(
    merged_m,
    left_on=['Season', '1TeamID'],
    right_on=['Season', 'TeamID'],
    how='left',
    suffixes=('', '_1')
)

# merge dla 2TeamID
submission_m = submission_m.merge(
    merged_m,
    left_on=['Season', '2TeamID'],
    right_on=['Season', 'TeamID'],
    how='left',
    suffixes=('', '_2')
)

In [47]:
# merge dla 1TeamID
submission_w = submission_w.merge(
    merged_w,
    left_on=['Season', '1TeamID'],
    right_on=['Season', 'TeamID'],
    how='left',
    suffixes=('', '_1')
)

# merge dla 2TeamID
submission_w = submission_w.merge(
    merged_w,
    left_on=['Season', '2TeamID'],
    right_on=['Season', 'TeamID'],
    how='left',
    suffixes=('', '_2')
)

In [48]:
for col in ['Seed', 'Seed_2']:
    submission_m[col] = (
        submission_m[col]
        .str.extract(r'(\d+)')      # wyciąga cyfry
        .astype(float)              # żeby NaN działał
        .fillna(17)                 # brak seed -> 17
        .astype(int)
    )
    submission_w[col] = (
        submission_w[col]
        .str.extract(r'(\d+)')      # wyciąga cyfry
        .astype(float)              # żeby NaN działał
        .fillna(17)                 # brak seed -> 17
        .astype(int)
    )

    # normalizacja + odwrócenie
    submission_m[col] = (17 - submission_m[col]) / 16
    submission_w[col] = (17 - submission_w[col]) / 16

In [49]:

submission_m.drop(columns=['Pred', 'TeamID'], inplace=True)

In [50]:
submission_w.drop(columns=['Pred', 'TeamID'], inplace=True)

In [51]:
total_m = m_tourney_full[m_tourney_full['Season'] <= 2025]
total_w = w_tourney_full[w_tourney_full['Season'] <= 2025]

In [ ]:
total_m.to_csv('total_m.csv', index=False)
total_w.to_csv('total_w.csv', index=False)


In [53]:
df1_m = submission_m.iloc[:len(submission_m)//2].copy()
df2_m = submission_m.iloc[len(submission_m)//2:].copy()
df1_w = submission_w.iloc[:len(submission_w)//2].copy()
df2_w = submission_w.iloc[len(submission_w)//2:].copy()

In [54]:
df1_m.to_csv('predict_m_1.csv', index=False)
df2_m.to_csv('predict_m_2.csv', index=False)
df1_w.to_csv('predict_w_1.csv', index=False)
df2_w.to_csv('predict_w_2.csv', index=False)